# Product QnA Chatbot

A **ReAct agent** that answers questions about laptops by combining three capabilities:

📄 **Architecture diagram & summary:** [README_PRODUCT_QNA.md](../README_PRODUCT_QNA.md)

---

## What This Example Demonstrates

| Component | Purpose |
|-----------|---------|
| **RAG (Retrieval-Augmented Generation)** | Product features from a vector store — the agent searches descriptions when asked about specs, CPU, memory, etc. |
| **Pricing tool** | CSV lookup — returns price for a laptop name (substring match). Returns -1 if not found. |
| **Conversation memory (MemorySaver)** | Keeps chat history per `thread_id`. "How much does it cost?" correctly refers to the last laptop discussed. |

---

## How It Works

1. **User asks** (e.g. "What are the features and pricing for GammaAir?")
2. **Agent reasons** — Decides which tools to call. May call both `get_product_features` and `get_laptop_price` in one turn.
3. **Tools execute** — RAG retrieves relevant chunks; pricing tool looks up the CSV.
4. **Agent synthesizes** — Combines tool results into a natural answer.

**Multi-turn:** Same `thread_id` = same conversation. "How much does it cost?" (after discussing SpectraBook) returns SpectraBook's price.

**Multi-user:** Different `thread_id` = separate conversations. USER 1 and USER 2 each have their own context.

---

## Tools

- **`get_laptop_price(laptop_name)`** — Substring match on product name → price. Returns -1 if no match.
- **`get_product_features`** — Vector search over product descriptions. Use for features, specs, available models.

---

## Data

- `data/laptop_pricing.csv` — Name, Price, ShippingDays
- `data/laptop_descriptions.txt` — Product descriptions (chunked and embedded in Chroma)

**See:** [`example8_product_qna_agent.py`](example8_product_qna_agent.py) for the full script with inline comments.

## 1. Setup Models

We need two OpenAI components:

- **ChatOpenAI** — The LLM that powers the agent. It reasons, decides which tools to call, and synthesizes answers. Uses `OPENAI_API_KEY` from `.env`.
- **OpenAIEmbeddings** — Converts text to vectors for the RAG retriever. When we chunk product descriptions and store them in Chroma, each chunk is embedded. Queries are also embedded so we can find the most similar chunks by vector similarity.

In [11]:
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

# LLM for the agent; embeddings for the RAG retriever
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

## 2. Product Pricing Tool

A simple `@tool` that looks up laptop prices from a CSV.

**How it works:**
- Loads `laptop_pricing.csv` into a pandas DataFrame.
- Uses `str.contains("^" + laptop_name, case=False, regex=True)` for a **substring match** at the start of the product name. So "GammaAir" matches "GammaAir X", "alpha" matches "AlphaBook Pro".
- Returns the price (int) if found, **-1** if no match. The agent can then say "I don't have that product."

**Why -1?** A clear sentinel value the LLM can interpret. The agent learns from the tool description and examples that -1 means "not found."

In [12]:
import pandas as pd
from langchain_core.tools import tool

DATA_DIR = Path("../data")
product_pricing_df = pd.read_csv(DATA_DIR / "laptop_pricing.csv")
print(product_pricing_df)

@tool
def get_laptop_price(laptop_name: str) -> int:
    """Return the price of a laptop given its name. Substring match.
    Returns -1 if no match."""
    match = product_pricing_df[product_pricing_df["Name"].str.contains("^" + laptop_name, case=False, regex=True)]
    return -1 if len(match) == 0 else int(match["Price"].iloc[0])

print("SpectraBook:: " + str(get_laptop_price.invoke({"laptop_name": "GammaAir"})))

            Name  Price  ShippingDays
0  AlphaBook Pro   1499             2
1     GammaAir X   1399             7
2  SpectraBook S   2499             7
3   OmegaPro G17   2199            14
4  NanoEdge Flex   1699             2
SpectraBook:: 1399


## 3. Product Features Retrieval Tool (RAG)

**RAG** = Retrieval-Augmented Generation. Instead of relying on the LLM's training data, we give it relevant chunks from our own documents at query time.

**Pipeline:**
1. **TextLoader** — Loads `laptop_descriptions.txt` into LangChain `Document` objects.
2. **RecursiveCharacterTextSplitter** — Splits documents into chunks (512 chars, 128 overlap). Overlap preserves context at chunk boundaries.
3. **Chroma.from_documents** — Embeds each chunk and stores in an in-memory vector DB. Similar chunks are "close" in vector space.
4. **create_retriever_tool** — Wraps `store.as_retriever(search_kwargs={"k": 2})` as a tool. When the LLM calls it with a query, we embed the query, find the top 2 most similar chunks, and return them as the tool's response.

**When the LLM calls it:** For questions about features, specs, CPU, memory, storage, design, or "Which laptops do you have?"

In [13]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.tools.retriever import create_retriever_tool

loader = TextLoader(str(DATA_DIR / "laptop_descriptions.txt"), encoding="utf-8")
docs = loader.load()
splits = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=128).split_documents(docs)

prod_feature_store = Chroma.from_documents(documents=splits, embedding=embedding)

get_product_features = create_retriever_tool(
    prod_feature_store.as_retriever(search_kwargs={"k": 2}),
    name="get_product_features",
    description="Search for laptop features and specs. Use when asked about CPU, memory, storage, design, or available laptops.",
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


## 4. Create Product QnA Agent

**create_react_agent** — One-liner that builds a ReAct graph: agent ↔ tools loop, with routing based on `tool_calls`.

**Key parameters:**
- **state_modifier** — System prompt prepended to every LLM call. "Use ONLY tools" prevents hallucination; "Handle small talk" lets it respond to "Hello" without calling tools.
- **checkpointer=MemorySaver()** — Enables **conversation memory**. Each `thread_id` gets its own checkpoint. When we invoke with the same `config`, the agent sees all previous messages in that thread. So "How much does it cost?" (after discussing SpectraBook) correctly refers to SpectraBook — the agent has the full history in context.

In [14]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

system_prompt = SystemMessage(content="""You are a professional chatbot for laptop questions.
Use ONLY the available tools for product info. Handle small talk professionally.""")

tools = [get_laptop_price, get_product_features]
checkpointer = MemorySaver()

product_qna_agent = create_react_agent(
    model=model,
    tools=tools,
    state_modifier=system_prompt,
    checkpointer=checkpointer,
    debug=False,
)

---

## Exercise: Execute the Product Q&A Chatbot

Let's execute the product QnA chatbot with a list of back-and-forth conversations. We first create a list of user messages, simulating a complete conversation, including greetings and follow-up questions. We set up a unique **thread ID** as an identifier for this conversation. We then send each message to the chatbot and print the responses.

**Run the code below and review the results.**

### 5. Single Query (Stream)

**stream()** with `stream_mode="values"` — Yields the full state after each step. We see each message as it's added: HumanMessage → AIMessage (with tool_calls) → ToolMessage(s) → AIMessage (final answer).

**message.pretty_print()** — Formats each message in a readable way. Useful for debugging the ReAct loop.

In [15]:
import uuid

config = {"configurable": {"thread_id": str(uuid.uuid4())}}
inputs = {"messages": [HumanMessage(content="What are the features and pricing for GammaAir?")]}

for stream in product_qna_agent.stream(inputs, config, stream_mode="values"):
    msg = stream["messages"][-1]
    if isinstance(msg, tuple):
        print(msg)
    else:
        msg.pretty_print()

================================ Human Message =================================

What are the features and pricing for GammaAir?
================================== Ai Message ==================================
Tool Calls:
  get_product_features (call_3QXkN3VCnxgG5msGdjBIKgoJ)
 Call ID: call_3QXkN3VCnxgG5msGdjBIKgoJ
  Args:
    query: GammaAir
  get_laptop_price (call_6qAiYBfxzT8ZmQzmXEKpIAUD)
 Call ID: call_6qAiYBfxzT8ZmQzmXEKpIAUD
  Args:
    laptop_name: GammaAir


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


================================= Tool Message =================================
Name: get_laptop_price

1399
================================== Ai Message ==================================

The **GammaAir X** features the following specifications:

- **Processor**: AMD Ryzen 7
- **Memory**: 32GB of DDR4 RAM
- **Storage**: 512GB NVMe SSD
- **Design**: Thin and light form factor, ideal for high performance in a portable design.

The price for the GammaAir is **$1399**. 

If you have any more questions or need further assistance, feel free to ask!


### 6. Multi-Turn Conversation

We create a list of user messages simulating a complete conversation: greetings, product questions, and follow-ups. We use the same `thread_id` for all messages so the agent sees the full history.

**Review the results:**

1. **Hello** — The chatbot responds with an appropriate greeting and its purpose. **No tool used** here; the agent handles small talk directly.

2. **I am looking to buy a laptop** — The chatbot responds with a menu of questions it can answer about laptops. Still no tool.

3. **Give me a list of available laptop names** — The agent calls the **retriever tool** (`get_product_features`) and returns the list from our product descriptions.

4. **Tell me about the features of SpectraBook** — The chatbot provides details for SpectraBook S (CPU, memory, storage, use case). Uses the retriever tool.

5. **How much does it cost?** — A follow-up question with no laptop name. The chatbot has **conversation memory** — it references the previous turn in context and correctly identifies SpectraBook. It then calls `get_laptop_price("SpectraBook")` to return $2,499.

6. **Thanks for the help** — The chatbot responds accordingly. No tool.

**This is an example of a real-world conversation.** The ability of the agent to understand the user question and pick the right tool (or no tool) enriches the conversation.

In [18]:
user_inputs = [
    "Hello",
    "I am looking to buy a laptop",
    "Give me a list of available laptop names",
    "Tell me about the features of SpectraBook",
    "How much does it cost?",
    "Thanks for the help",
]

config = {"configurable": {"thread_id": str(uuid.uuid4())}}

for user_input in user_inputs:
    print(f"\n{'─' * 40}\nUSER : {user_input}")
    result = product_qna_agent.invoke({"messages": [HumanMessage(content=user_input)]}, config=config)
    print(f"AGENT : {result['messages'][-1].content}")


────────────────────────────────────────
USER : Hello
AGENT : Hello! How can I assist you with your laptop questions today?

────────────────────────────────────────
USER : I am looking to buy a laptop
AGENT : Great! What specific features or specifications are you looking for in a laptop? For example, do you have preferences regarding the CPU, memory, storage, or design?

────────────────────────────────────────
USER : Give me a list of available laptop names
AGENT : Here are some available laptops you might consider:

1. **NanoEdge Flex**: 
   - Compact 13-inch laptop
   - Intel i5 processor
   - 8GB RAM
   - 256GB SSD
   - Ultra-portable with a flexible hinge design for tablet mode.

2. **AlphaBook Pro**: 
   - Sleek ultrabook
   - 12th Gen Intel i7 processor
   - 16GB DDR4 RAM
   - 1TB SSD
   - Thin and light design with a 14-inch display, ideal for professionals on the go.

3. **GammaAir X**: 
   - AMD Ryzen 7 processor
   - 32GB DDR4 memory
   - 512GB NVMe SSD
   - Perfect for h

---

## Exercise: Test Conversation Memory

Let's test how the conversation memory of the chatbot works. We create **two conversations**, send the same follow-up question to both, and see how the chatbot uses its memory to correctly identify each user's laptop.

**Setup:** We create a function `execute_prompt` that takes a user identifier, a config object, and a prompt. It executes the prompt using the `thread_id` in the config and prints the results. We create two threads and corresponding config objects — simulating two users using the chatbot concurrently.

In [19]:
def execute_prompt(user_label: str, config: dict, prompt: str):
    """Execute a prompt using the thread_id in config and print the result."""
    result = product_qna_agent.invoke({"messages": [HumanMessage(content=prompt)]}, config=config)
    print(f"\n{user_label}: {result['messages'][-1].content}")

# Create two threads — simulates two users using the chatbot concurrently
config_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}
config_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}

# Execute: USER 1 asks about SpectraBook, USER 2 about GammaAir
execute_prompt("USER 1", config_1, "Tell me about the features of SpectraBook")
execute_prompt("USER 2", config_2, "Tell me about the features of GammaAir")
# Same follow-up question — each user gets the price for their laptop
execute_prompt("USER 1", config_1, "What is its price?")
execute_prompt("USER 2", config_2, "What is its price?")


USER 1: The SpectraBook S is designed for power users and comes with impressive features:

- **Processor**: Intel Core i9
- **Memory**: 64GB RAM
- **Storage**: 2TB SSD
- **Use Case**: Ideal for intensive tasks such as video editing and 3D rendering, making it a workstation-class laptop.

If you have any more questions or need information on other laptops, feel free to ask!

USER 2: It seems there isn't a specific model named "GammaAir," but there is a model called "GammaAir X." Here are its features:

- **Processor**: AMD Ryzen 7
- **Memory**: 32GB of DDR4 RAM
- **Storage**: 512GB NVMe SSD
- **Design**: Thin and light form factor, making it portable
- **Performance**: Ideal for high-performance tasks

If you need information on other models or specific features, feel free to ask!

USER 1: The price of the SpectraBook S is $2499. If you have any more questions or need further assistance, just let me know!

USER 2: The price of the GammaAir X is $1,399. If you have any more questions or

**Review the outputs:**

- The chatbot answers about the features for **SpectraBook** (USER 1) and **GammaAir** (USER 2) correctly to the corresponding users.
- For the follow-up question **"What is its price?"**, the chatbot uses **conversation memory** to correctly identify the laptop each user is asking about:
  - USER 1 gets SpectraBook's price ($2,499)
  - USER 2 gets GammaAir's price ($1,399)

**This completes our example for creating a ReAct chatbot using pre-built functions.** In the next chapter, we will create a custom chatbot using LangGraph.